## Initial setup

Before running the notebook:
- update the .env file with your team's `OPENAI_API_KEY`
- Install uv `pip install uv`
- run `uv sync`
- run `uv run phoenix serve` (If Arize Phoenix is enabled) - takes a few minutes
- Select Kernal for this notebook

## Raven Starter Bot

This notebook contains a **Raven Starter Bot** with:

1. **Message parsing** ( `parse_message`, `parse_outgoing_message`)
2. **Role-specific random responders** (Villager, Raven, Detective, Doctor)
4. **A single async loop** `connect_parse_respond_forever()`

You can:
- Run the bot and let it play matches.
- Inspect each game later by `gameId` using the log utilities.
- Stop the bot




## Imports and basic configuration

In [192]:
import asyncio
import json
import os
import random
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from collections import defaultdict
import itertools
from datetime import datetime
from pathlib import Path

import websockets

from openai import AsyncOpenAI
from pydantic import BaseModel
from uuid import uuid4
from dotenv import load_dotenv


load_dotenv()

# --- WebSocket configuration ---
WS_URL = os.getenv("WS_URL", "ws://localhost:2025")
CONNECT_TIMEOUT = 10 # seconds to wait when opening
RECV_TIMEOUT = 0.1
KEEP_ALIVE = True
ENABLE_PHOENIX = True


def ts() -> str:
    """Return a human-readable timestamp for logging."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


## Initialize Phoenix

In [193]:
if ENABLE_PHOENIX:

    PHOENIX_PROJECT_NAME = "raven-bot" + '-' + ts()

    try:
        from openinference.instrumentation.openai import OpenAIInstrumentor
        from phoenix.otel import register

        tracer_provider = register(project_name=PHOENIX_PROJECT_NAME, protocol="http/protobuf")
        OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

    except ImportError:
        print("Phoenix OpenTelemetry instrumentation is not installed. please run 'uv sync'")

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


OpenTelemetry Tracing Details
|  Phoenix Project: raven-bot-2025-11-28 18:34:24
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Hello LLM

Run Below cell to make sure your OPENAI_API_KEY is working

In [194]:
no_of_tips = 5
max_characters_per_tip = 140

if not os.environ.get("OPENAI_API_KEY"):
    print(f"⚠️ OPENAI_API_KEY was not found in the environment.")

# Single async OpenAI client
client = AsyncOpenAI()

MODEL_NAME = "gpt-5-nano"
REASONING_EFFORT = "low"   # "minimal""low" / "medium" / "high"
VERBOSITY = "low"             # keep answers short/direct

try:
    user_prompt = (
        f"I'm playing a game of Mafia (Raven) in a 24 Hour AI hackathon."
        f"Give me {no_of_tips} tips to help me win the hackathon."
        f"Keep each tip less than {max_characters_per_tip} characters."
    )
    # user_prompt = "Say Hi"
    resp = await client.responses.parse(
        model=MODEL_NAME,
        input=[ { "role": "user", "content": user_prompt,}],
        reasoning={"effort": REASONING_EFFORT},
        text={"verbosity": VERBOSITY}
    )
    print(resp.output_text)

except Exception as e:
    print(f"Error during OpenAI API call: {e}")
    raise

Exception while exporting Span.
Traceback (most recent call last):
  File "d:\SPL\raven_starter_bot_v1.0\.venv\Lib\site-packages\urllib3\connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\SPL\raven_starter_bot_v1.0\.venv\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "d:\SPL\raven_starter_bot_v1.0\.venv\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\SPL\raven_starter_bot_v1.0\.venv\Lib\site-packages\urllib3\connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "d:\SPL\raven_starter_bot_v1.0\.venv\Lib\site-packages\

- Define a clear MVP: core features and fastest path to win.
- Time-box tasks: 2–4 hour sprints, track blockers, avoid scope creep.
- Communicate progress: brief updates, blockers, decisions, data quality.
- Build testable experiments: logs, reproducible runs, compare baselines.
- Collaborate smartly: assign roles, avoid duplicate work, merge early.


# Helper functions

## Message model – `ParsedMessage`

In [195]:

@dataclass
class ParsedMessage:
    raw: str                                 # original JSON string
    type: Optional[str] = None               # message type

    # Core identifiers
    match_id: Optional[str] = None
    game_id: Optional[str] = None
    your_id: Optional[str] = None

    # Game and phase context
    day: Optional[int] = None
    phase: Optional[str] = None              # "morning" or "night"
    timeout: Optional[float] = None          # generic timeout (seconds)
    first_vote_timeout: Optional[float] = None
    time_remaining: Optional[float] = None   # used in *_player-comment, raven-comment

    # GAME START info
    your_role: Optional[str] = None
    raven_count: Optional[int] = None
    detective_count: Optional[int] = None
    doctor_count: Optional[int] = None
    villager_count: Optional[int] = None

    # Common interaction fields
    otp: Optional[str] = None
    comment: Optional[str] = None

    # Player lists / status
    all_players: List[Dict[str, Any]] = field(default_factory=list)
    villagers_alive: List[str] = field(default_factory=list)
    players_alive: List[str] = field(default_factory=list)
    discussions: List[Dict[str, Any]] = field(default_factory=list)

    # Voting information
    votes: List[str] = field(default_factory=list)   # unified votes field
    done_voting: Optional[bool] = None
    player_lynched: Optional[str] = None

    # Role-identification info (Detective / shared)
    identified_raven: List[str] = field(default_factory=list)
    identified_villager: List[str] = field(default_factory=list)

    # Comment metadata
    player_id: Optional[str] = None
    llm_model_used: Optional[str] = None

    # Ack / investigation results
    request_status: Optional[str] = None             # for generic ack
    investigated: List[str] = field(default_factory=list)  # for ack-night-investigation
    is_raven: List[bool] = field(default_factory=list)     # parallel to investigated

    # Game result
    result: Optional[str] = None


## 🔧 Function:  `parse_message` to parse incoming messages

In [196]:
def parse_message(msg: str) -> Optional[ParsedMessage]:
    """Parse a raw JSON string from the server into a ParsedMessage object.

    This is aligned with the latest singham protocol.md and the observed
    JSONL logs. Legacy fields like a top-level ``vote`` on incoming
    messages are no longer supported here.
    """
    try:
        data = json.loads(msg)
    except Exception as e:
        print("❌ Invalid JSON:", msg)
        print("Error:", e)
        return None

    p = ParsedMessage(raw=msg)

    # Common fields present on most messages
    p.type = data.get("type")
    p.match_id = data.get("matchId")
    p.game_id = data.get("gameId")
    p.your_id = data.get("yourId")
    p.otp = data.get("otp")
    p.day = data.get("day")
    p.phase = data.get("phase")

    # Timers
    if "timeout" in data:
        p.timeout = data.get("timeout")
    if "firstVoteTimeout" in data:
        p.first_vote_timeout = data.get("firstVoteTimeout")
    if "timeRemaining" in data:
        p.time_remaining = data.get("timeRemaining")

    # 1) GAME START
    if p.type == "game-start":
        p.raven_count = data.get("ravenCount")
        p.detective_count = data.get("detectiveCount")
        p.doctor_count = data.get("doctorCount")
        p.villager_count = data.get("villagerCount")
        p.your_role = data.get("yourRole")
        return p

    # 2) PLAYER STATUS
    if p.type == "player-status":
        # allPlayers: [{ id, isAlive?, lynchedBy, lynchedDay }, ...]
        p.all_players = data.get("allPlayers", [])
        return p

    # 3) PHASE RESULT
    if p.type == "phase-result":
        # playerLynched may be empty (no one lynched)
        lynched = data.get("playerLynched")
        p.player_lynched = lynched or None
        return p

    # 4) GENERIC ACK
    if p.type == "ack":
        p.request_status = data.get("requestStatus")
        return p

    # 5) NIGHT DISCUSSION (Ravens)
    if p.type == "night-discussion":
        p.villagers_alive = data.get("villagersAlive", [])
        return p

    # 6) RAVEN COMMENT (Server → Ravens)
    if p.type == "raven-comment":
        p.discussions = data.get("discussions", [])
        if p.discussions:
            last = p.discussions[-1]
            p.player_id = last.get("playerId")
            p.comment = last.get("comment")
            p.votes = list(last.get("votes", []))
        return p

    # 7) MORNING DISCUSSION (Villagers)
    if p.type == "morning-discussion":
        p.players_alive = data.get("playersAlive", [])
        return p

    # 8) MORNING PLAYER COMMENT (Server → Villagers)
    if p.type == "morning-player-comment":
        p.discussions = data.get("discussions", [])
        if p.discussions:
            last = p.discussions[-1]
            p.player_id = last.get("playerId")
            p.comment = last.get("comment")
            p.votes = list(last.get("votes", []))
        return p

    # 9) NIGHT INVESTIGATION (Detective)
    if p.type == "night-investigation":
        p.players_alive = data.get("playersAlive", [])
        p.identified_raven = data.get("identifiedRavens", []) or []
        p.identified_villager = data.get("identifiedVillagers", []) or []
        return p

    # 10) NIGHT PROTECTION (Doctor)
    if p.type == "night-protection":
        p.players_alive = data.get("playersAlive", [])
        return p

    # 11) ACK for detective investigation
    if p.type == "ack-night-investigation":
        p.investigated = data.get("investigated", []) or []
        p.is_raven = data.get("isRaven", []) or []
        return p

    # 12) GAME RESULT (if/when used)
    if p.type == "game-result":
        p.result = data.get("result")  # "won", "lost", "draw", etc.
        return p

    # Fallback: unknown type – still return ParsedMessage for debugging
    return p


## 🔧 Functions: `parse_outgoing_message`

In [197]:
def parse_outgoing_message(raw: str) -> ParsedMessage:
    """Parse the bot → server JSON into a ParsedMessage for logging.

    Outgoing messages now always use the ``votes`` list (no legacy ``vote`` key).
    """
    try:
        data = json.loads(raw)
    except Exception:
        # log as raw, but still return ParsedMessage
        return ParsedMessage(raw=raw)

    p = ParsedMessage(raw=raw)
    p.type = data.get("type")
    p.game_id = data.get("gameId")
    p.your_id = data.get("yourId")
    p.otp = data.get("otp")
    p.comment = data.get("comment")

    # Voting-related fields (used by all roles)
    p.votes = data.get("votes", []) or []
    p.done_voting = data.get("doneVoting")
    p.llm_model_used = data.get("llmModelUsed")

    return p

## 🔧 Functions: `print_parsed_message`

In [198]:
def print_parsed_message(p: Optional[ParsedMessage]) -> None:
    """Pretty-print the key fields of a ParsedMessage for debugging."""
    if p is None:
        print("❌ Nothing to print (parsed is None)")
        return
    
    

    print("\n================= 🧩 PARSED MESSAGE =================")
    print(f"Type           : {p.type}")
    print(f"Match ID       : {p.match_id}")
    print(f"Game ID        : {p.game_id}")
    print(f"Your ID        : {p.your_id}")
    if p.day is not None:
        print(f"Day            : {p.day}")
    if p.phase is not None:
        print(f"Phase          : {p.phase}")
    if p.timeout is not None:
        print(f"Timeout (s)    : {p.timeout}")
    if p.first_vote_timeout is not None:
        print(f"First Vote TO  : {p.first_vote_timeout}")
    if p.time_remaining is not None:
        print(f"Time Remaining : {p.time_remaining}")
    if p.otp:
        print(f"OTP            : {p.otp}")

    # GAME START
    if p.type == "game-start":
        print(f"Your Role      : {p.your_role}")
        print(f"Ravens         : {p.raven_count}")
        print(f"Detectives     : {p.detective_count}")
        print(f"Doctors        : {p.doctor_count}")
        print(f"Villagers      : {p.villager_count}")

    # PLAYER STATUS
    elif p.type == "player-status":
        print("All Players    :")
        for pl in p.all_players:
            print(f"  - {pl}")

    # PHASE RESULT
    elif p.type == "phase-result":
        print(f"Player Lynched : {p.player_lynched}")

    # ACK
    elif p.type == "ack":
        print(f"Request Status : {p.request_status}")

    # NIGHT DISCUSSION
    elif p.type == "night-discussion":
        print(f"Villagers Alive: {p.villagers_alive}")

    # RAVEN COMMENT
    elif p.type == "raven-comment":
        print("Discussions    :")
        for d in p.discussions:
            print(f"  - {d.get('playerId')}: {d.get('comment')} (votes={d.get('votes')})")
        if p.votes:
            print(f"Last Votes     : {p.votes}")

    # MORNING DISCUSSION
    elif p.type == "morning-discussion":
        print(f"Players Alive  : {p.players_alive}")

    # MORNING PLAYER COMMENT
    elif p.type == "morning-player-comment":
        print("Discussions    :")
        for d in p.discussions:
            print(f"  - {d.get('playerId')}: {d.get('comment')} (votes={d.get('votes')})")
        if p.votes:
            print(f"Last Votes     : {p.votes}")

    # NIGHT INVESTIGATION
    elif p.type == "night-investigation":
        print(f"Players Alive  : {p.players_alive}")
        print(f"Identified Ravens   : {p.identified_raven}")
        print(f"Identified Villagers: {p.identified_villager}")

    # NIGHT PROTECTION
    elif p.type == "night-protection":
        print(f"Players Alive  : {p.players_alive}")

    # ACK NIGHT INVESTIGATION
    elif p.type == "ack-night-investigation":
        print(f"Investigated   : {p.investigated}")
        print(f"Is Raven       : {p.is_raven}")

    # GAME RESULT
    elif p.type == "game-result":
        print(f"Game Result    : {p.result}")

    else:
        print("⚠️ No specific printer for this type; showing raw dict:")
        try:
            print(json.loads(p.raw))
        except Exception:
            print(p.raw)

    print("=====================================================")



# Role-specific responders (Villager, Raven, Detective, Doctor)

###  All Roles: `build_vote_from_morning_discussion`

In [199]:
MORNING_MSG_COUNT: Dict[str, int] = defaultdict(int)

async def build_vote_from_morning_discussion(parsed: ParsedMessage, done_voting: bool) -> Dict[str, Any]:
    """
    Villager: build a vote from a morning-discussion message.
    Now uses OpenAI to choose a target + comment, with a simple random fallback.
    """
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    alive = parsed.players_alive or []
    possible_targets = [p for p in alive if p != your_id]

    vote_target = random.choice(possible_targets) if possible_targets else None
    comment = f"I think {vote_target} may be suspicious. Casting my vote."

    votes = [vote_target] if vote_target is not None else []

    msg: Dict[str, Any] = {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
    }

    if done_voting:
        msg["doneVoting"] = True

    return msg


### Raven: `build_raven_vote_from_night_discussion`

In [200]:
async def build_raven_vote_from_night_discussion(parsed: ParsedMessage) -> Dict[str, Any]:
    """
    Raven: build a vote message from a night-discussion message.

    Uses LLM to pick villagers to target (votes list).
    If LLM fails or gives nonsense, fallback = all villagers, sorted.
    """
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    villagers = sorted(parsed.villagers_alive or [])

    votes: List[str] = []
    if not votes and villagers:
        votes = villagers[:] 
        comment = (
            "As Raven, I am voting to eliminate the following villagers: "
            + ", ".join(votes)
        )
    elif not villagers:
        votes = []
        comment = "As Raven, I have no villagers to target."

    return {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
    }


### Detective: `build_detective_vote_from_night_investigation`

In [201]:
async def build_detective_vote_from_night_investigation(parsed: ParsedMessage) -> Dict[str, Any]:
    """Detective: build a vote from a night-investigation message."""
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    alive = parsed.players_alive or []
    safe_players = set(parsed.identified_villager or [])

    possible_targets = [p for p in alive if p not in safe_players]
    if not possible_targets:
        possible_targets = alive[:]

    
    vote_target = random.choice(possible_targets) if possible_targets else None
    comment = f"As Detective, I want to investigate {vote_target}. Casting my vote on them."

    votes = [vote_target] if vote_target is not None else []

    return {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
    }


### Doctor: `build_doctor_vote_from_night_protection`

In [202]:
async def build_doctor_vote_from_night_protection(parsed: ParsedMessage) -> Dict[str, Any]:
    """Doctor: build a protection vote from a night-protection message."""
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    alive = parsed.players_alive or []

    protect_target = random.choice(alive) if alive else None
    comment = f"As Doctor, I choose to protect {protect_target} tonight."

    votes = [protect_target] if protect_target is not None else []

    return {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
    }


## Logging – in-memory + JSONL file

In [203]:
# In-memory game logs: game_id -> list of events
GAME_LOGS: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

# Global event counter (across the whole run)
EVENT_COUNTER = itertools.count(1)

# Log file setup
LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE_PATH = os.path.join(
    LOG_DIR,
    f"raven_events_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jsonl"
)

print(f"Logging events to: {LOG_FILE_PATH}")


def record_event(*, direction: str, raw: str, parsed: Optional[ParsedMessage] = None) -> None:
    """Record a single event both in memory and in the JSONL log file."""
    seq = next(EVENT_COUNTER)
    timestamp = ts()

    game_id = getattr(parsed, "game_id", None) if parsed is not None else None
    match_id = getattr(parsed, "match_id", None) if parsed is not None else None
    msg_type = getattr(parsed, "type", None) if parsed is not None else None

    if not game_id:
        game_id = "__no_game_id__"

    event = {
        "seq": seq,
        "ts": timestamp,
        "direction": direction,  # "IN" or "OUT"
        "type": msg_type,
        "matchId": match_id,
        "gameId": game_id,
        "raw": raw,
    }

    # 1) In-memory
    GAME_LOGS[game_id].append(event)

    # 2) On disk (append JSONL)
    with open(LOG_FILE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(event) + "\n")


def print_game_log_from_memory(game_id: str) -> None:
    """Print all events for a gameId from in-memory logs."""
    events = GAME_LOGS.get(game_id, [])
    if not events:
        print(f"No events logged for gameId={game_id}")
        return

    print("\n" + "=" * 80)
    print(f"🎮 GAME LOG (in-memory) for gameId={game_id}")
    print("=" * 80)

    for ev in events:
        direction = "Server → Bot" if ev["direction"] == "IN" else "Bot → Server"
        print(
            f"\n#{ev['seq']} [{ev['ts']}] {direction} "
            f"(type={ev['type']}, matchId={ev['matchId']})"
        )
        print("-" * 80)
        try:
            obj = json.loads(ev["raw"])
            print(json.dumps(obj, indent=2))
        except Exception:
            print(ev["raw"])

    print("\n" + "=" * 80 + "\n")

Logging events to: logs\raven_events_20251128_183437.jsonl


# Main async loop – connect, parse, respond, log

### How the non-blocking LLM logic works

This section is the **heart of the bot's concurrency model**. The goal is:

> Keep listening to the server all the time, without waiting for the LLM.

We do this in two pieces:

1. **`connect_parse_respond_forever` (main loop)**  
   - Opens the WebSocket connection to the game server.  
   - Repeatedly does `ws.recv()` to get the next message.  
   - Parses and logs the message.  
   - For messages that need an LLM decision, it calls:

     ```python
     asyncio.create_task(handle_message_and_respond(parsed, ws, send_lock))
     ```

   - This line *starts* an async task and immediately returns, so the loop
     can go back to waiting for the next message.

2. **`handle_message_and_respond` (background task)**  
   - Runs **in the background**, one task per message that needs the LLM.  
   - Decides what to do based on `parsed.type` (Villager / Raven / Detective / Doctor).  
   - Calls the appropriate async helper which internally talks to OpenAI.  
   - Builds the outgoing JSON payload, logs it, and sends it via `ws.send(...)`.  
   - Uses a shared `send_lock` so that only one task writes to the WebSocket at a time.

Because of this design:

- The bot can receive and handle **multiple games or phases in parallel**.
- A slow LLM call does **not** block the WebSocket; other messages keep flowing.
- The pattern is simple and beginner-friendly: one main loop + one background handler.


## Handler `handle_message_and_respond`

In [204]:
class GameMetadata:
  def __init__(self, match_id: Optional[str] = None, game_id: Optional[str] = None,
         raven_count: Optional[int] = None, detective_count: Optional[int] = None,
         doctor_count: Optional[int] = None, villager_count: Optional[int] = None):
    self.match_id = match_id
    self.game_id = game_id
    self.raven_count = raven_count
    self.detective_count = detective_count
    self.doctor_count = doctor_count
    self.villager_count = villager_count
  
  def print_values(self) -> None:
    for name, value in vars(self).items():
      print(f"{name}: {value}")

In [205]:
class PlayerMetadata:
  def __init__(self, your_id: str, your_role: str, team: Optional[str] = None):
    self.your_id = your_id
    self.your_role = your_role
  
  def print_values(self) -> None:
    for name, value in vars(self).items():
      print(f"{name}: {value}")

In [206]:
class GameState:
    def __init__(
        self,
        phase: Optional[str] = None,
        day: int = 0,
        players_alive: Optional[List[str]] = None,
        players_dead: Optional[Dict[str, Dict[str, Any]]] = None,
    ):
        """
        Simple game state holder.

        players_dead structure:
            {
            "player_id_1": {
                "killed_by": "player_x" | None,
                "killed_day": 2 | None,
                "killed_phase": "night" | "morning" | None
            },
            ...
            }
        """
        self.phase: Optional[str] = phase
        self.day: int = day
        self.players_alive: List[str] = players_alive if players_alive is not None else []
        self.players_dead: Dict[str, Dict[str, Any]] = (
            players_dead if players_dead is not None else {}
        )
    
    def print_values(self) -> None:
        for name, value in vars(self).items():
            print(f"{name}: {value}")
    

In [207]:
GAME_METADATA: Dict[str, GameMetadata] = {}
PLAYER_METADATA: Dict[str, PlayerMetadata] = {}
GAME_STATE: Dict[str, GameState] = {}

In [ ]:
async def handle_message_and_respond(
    parsed: ParsedMessage, ws, send_lock: asyncio.Lock
) -> None:
    """
    Handle a single parsed message **in a background task**.

    This function is where we decide whether we need to call the LLM and send
    a response back to the server. It is meant to be launched with
    `asyncio.create_task(...)`, so the main WebSocket receive loop does not
    wait for the LLM call to finish.

    Parameters
    ----------
    parsed : ParsedMessage
        The already-parsed server message.
    ws : websockets.WebSocketClientProtocol
        The live WebSocket connection back to the game server.
    send_lock : asyncio.Lock
        A lock so that only one task calls `ws.send(...)` at a time.
    """
    try:
        if parsed is None:
            return

        outgoing: Optional[Dict[str, Any]] = None

        if parsed.type == "game-start":
            game_metadata = GameMetadata(
                match_id=parsed.match_id,
                game_id=parsed.game_id,
                raven_count=parsed.raven_count,
                detective_count=parsed.detective_count,
                doctor_count=parsed.doctor_count,
                villager_count=parsed.villager_count,
            )
            player_metadata = PlayerMetadata(
                your_id=parsed.your_id, your_role=parsed.your_role
            )
            GAME_METADATA[parsed.game_id] = game_metadata
            PLAYER_METADATA[parsed.game_id] = player_metadata
            # Log current metadata dictionaries for debugging
            print(
                f"[{ts()}] GAME_METADATA {parsed.game_id} = {GAME_METADATA[parsed.game_id].print_values()}"
            )
            print(
                f"[{ts()}] PLAYER_METADATA {parsed.game_id} = {PLAYER_METADATA[parsed.game_id].print_values()}"
            )

        elif parsed.type == "player-status":
            # Construct dead_players from parsed.all_players
            players_alive = []
            players_dead = {}
            for player in parsed.all_players:
                player_id = player.get("id")
                if player_id is None:
                    continue

                if player.get("isAlive?") is True:
                    players_alive.append(player_id)
                else:
                    killed_by = player.get("lynchedBy")
                    killed_day = player.get("lynchedDay")
                    killed_phase = "morning" if killed_by == "Villager" else "night"
                    players_dead[player_id] = {
                        "killed_by": killed_by,
                        "killed_day": killed_day,
                        "killed_phase": killed_phase,
                    }

            game_state = GameState(
                phase=parsed.phase,
                day=parsed.day,
                players_alive=players_alive,
                players_dead=players_dead,
            )
            GAME_STATE[parsed.game_id] = game_state
            print(
                f"[{ts()}] GAME_STATE {parsed.game_id} = {GAME_STATE[parsed.game_id].print_values()}"
            )

        elif parsed.type == "morning-discussion":
            # All Players: build a vote for morning discussion
            outgoing = await build_vote_from_morning_discussion(
                parsed, done_voting=True
            )

        elif parsed.type == "night-discussion":
            # Raven: build votes for night elimination
            outgoing = await build_raven_vote_from_night_discussion(parsed)

        elif parsed.type == "night-investigation":
            # Detective: choose a target to investigate
            outgoing = await build_detective_vote_from_night_investigation(parsed)

        elif parsed.type == "night-protection":
            # Doctor: choose someone to protect
            outgoing = await build_doctor_vote_from_night_protection(parsed)

        else:
            # For all other message types (acks, results, etc.) we just log.
            print(f"[{ts()}] ℹ️ No action needed/handled for type={parsed.type!r}")
            return

        if not outgoing:
            # Nothing to send (e.g., no valid targets)
            print(f"[{ts()}] ℹ️ No outgoing message built for type={parsed.type!r}")
            return

        # --- Serialize and log OUT message ---
        raw_out = json.dumps(outgoing)
        out_parsed = parse_outgoing_message(raw_out)
        record_event(direction="OUT", raw=raw_out, parsed=out_parsed)

        # Only one task should call ws.send at a time → use a lock.
        async with send_lock:
            await ws.send(raw_out)

        print(f"[Bot → sent]\n{json.dumps(outgoing, indent=2)}")

    except Exception as e:
        # Make sure background task failures are visible
        print(f"[{ts()}] ⚠️ Error in handle_message_and_respond: {e}")

## Main Loop `connect_parse_respond_forever`

In [209]:
async def connect_parse_respond_forever():
    """
    Main loop: keep the WebSocket connection open, keep *listening* for messages,
    and spin off background tasks to handle any LLM / decision work.

    The key idea:
    - This loop ONLY waits on `ws.recv()` and other cheap operations.
    - For any message that needs an LLM call, we do:

          asyncio.create_task(handle_message_and_respond(parsed, ws, send_lock))

      so the loop can immediately go back to listening for the next message.
    """
    print(f"[{ts()}] 🔌 Connecting to {WS_URL} ...")
    try:
        async with websockets.connect(WS_URL, open_timeout=CONNECT_TIMEOUT) as ws:
            print(f"[{ts()}] ✅ Connection established.")

            # One lock shared by all background tasks that want to send on this websocket
            send_lock = asyncio.Lock()

            while True:
                try:
                    # 1) Wait for the next message from the server
                    msg = await asyncio.wait_for(ws.recv(), timeout=RECV_TIMEOUT)

                    # 2) Log and parse incoming message
                    print(f"\n[Server → raw] {msg}")
                    parsed = parse_message(msg)
                    # print_parsed_message(parsed)
                    print(f"Parsed Type: {parsed.type}")

                    # Record the IN event (even if parsing failed, we capture raw)
                    record_event(direction="IN", raw=msg, parsed=parsed)

                    if parsed is None:
                        # Invalid/unknown message, nothing more to do
                        continue

                    # 3) Fire-and-forget background task to handle LLM + response.
                    #    This is what makes the LLM call *non-blocking* for the main loop.
                    asyncio.create_task(handle_message_and_respond(parsed, ws, send_lock))

                except asyncio.TimeoutError:
                    if KEEP_ALIVE:
                        # No messages recently, but keep the connection open.
                        continue
                    print(f"[{ts()}] ⏹️ No messages within {RECV_TIMEOUT}s; closing connection.")
                    break
                except websockets.exceptions.ConnectionClosedOK:
                    print(f"[{ts()}] 🔒 Connection closed by server (OK).")
                    break
                except websockets.exceptions.ConnectionClosedError as e:
                    print(f"[{ts()}] ❌ Connection closed with error: {e}")
                    break
                except Exception as e:
                    print(f"[{ts()}] ⚠️ Unexpected error while listening: {e}")
                    break
    except Exception as e:
        print(f"[{ts()}] ❌ Connection failed: {e}")

# Run and Debug

## Run the bot

In [ ]:
#if the bot_task is already running, run bot_task.cancel() first the old task
try:
    bot_task.cancel()
except Exception:
    pass

# Start a (new) bot task
bot_task = asyncio.create_task(connect_parse_respond_forever())


[2025-11-28 18:34:37] 🔌 Connecting to ws://localhost:2025 ...
[2025-11-28 18:34:39] ✅ Connection established.

[Server → raw] {"matchId":"7DFC992E-D388-4EAE-91F0-633D70CCE94F","gameId":"71BA921E-BA91-4AED-96BF-878EC96E83B6","gameNumber":"Game1","yourId":"P4","type":"game-start","ravenCount":2,"detectiveCount":1,"doctorCount":1,"villagerCount":4,"yourRole":"Detective"}
Parsed Type: game-start
match_id: 7DFC992E-D388-4EAE-91F0-633D70CCE94F
game_id: 71BA921E-BA91-4AED-96BF-878EC96E83B6
raven_count: 2
detective_count: 1
doctor_count: 1
villager_count: 4
[2025-11-28 18:34:49] GAME_METADATA 71BA921E-BA91-4AED-96BF-878EC96E83B6 = None
your_id: P4
your_role: Detective
[2025-11-28 18:34:49] PLAYER_METADATA 71BA921E-BA91-4AED-96BF-878EC96E83B6 = None
[2025-11-28 18:34:49] ℹ️ No outgoing message built for type='game-start'

[Server → raw] {"gameId":"71BA921E-BA91-4AED-96BF-878EC96E83B6","gameNumber":"Game1","yourId":"P4","type":"player-status","day":0,"phase":"morning","allPlayers":[{"id":"P1","i

## Print Games logs

In [211]:
# game_ids = list(GAME_LOGS.keys())
# game_ids

In [212]:
# print_game_log_from_memory(game_ids[0])

## Run the Below cell stop the task

In [213]:
# bot_task.cancel()